# 深入探索Dataset和DataLoader

In [1]:
#|export
import pickle,gzip,math,os,time,shutil,torch,random
import fastcore.all as fc,matplotlib as mpl,numpy as np,matplotlib.pyplot as plt
from collections.abc import Mapping
from pathlib import Path
from operator import attrgetter,itemgetter
from functools import partial
from copy import copy
from contextlib import contextmanager
from scipy import linalg

from fastcore.foundation import L
import torchvision.transforms.functional as TF,torch.nn.functional as F
from torch import tensor,nn,optim
from torch.utils.data import DataLoader,default_collate
from torch.nn import init
from torch.optim import lr_scheduler
from torcheval.metrics import MulticlassAccuracy
from datasets import load_dataset,load_dataset_builder
import torch

from miniai.datasets import *
from miniai.conv import *
from miniai.learner import *
from miniai.activations import *
from miniai.init import *
from miniai.sgd import *
from miniai.resnet import *
from miniai.augment import *
from miniai.accel import *

/Users/bytedance/venv/lib/python3.13/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
from fastcore.test import test_close
from torch import distributions

torch.set_printoptions(precision=2, linewidth=140, sci_mode=False)
torch.manual_seed(1)
mpl.rcParams['image.cmap'] = 'gray_r'

import logging
logging.disable(logging.WARNING)

set_seed(42)
if fc.defaults.cpus>8: fc.defaults.cpus=8

In [3]:
xl,yl = 'image','label'
name = "fashion_mnist"
bs = 512

In [4]:
ds = load_dataset(name)

In [5]:
ds_train, ds_val = ds.get('train'), ds.get('test')
ds_train, ds_val

(Dataset({
     features: ['image', 'label'],
     num_rows: 60000
 }),
 Dataset({
     features: ['image', 'label'],
     num_rows: 10000
 }))

In [7]:
F.pad(TF.to_tensor(ds_train[xl][0]), (2, 2, 2, 2))[:,15:17,:]

tensor([[[0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.02, 0.00, 0.00, 0.22, 0.93, 0.89, 0.90, 0.89, 0.94, 0.91,
          0.84, 0.85, 0.87, 0.92, 0.85, 0.85, 0.82, 0.36, 0.00, 0.00, 0.00],
         [0.00, 0.00, 0.00, 0.00, 0.00, 0.02, 0.02, 0.03, 0.01, 0.00, 0.00, 0.00, 0.00, 0.00, 0.93, 0.89, 0.85, 0.87, 0.87, 0.86, 0.87,
          0.87, 0.85, 0.87, 0.90, 0.84, 0.85, 1.00, 0.30, 0.00, 0.00, 0.00]]])

In [8]:
@inplace
def transformi(b): b[xl] = [F.pad(TF.to_tensor(o), (2,2,2,2))*2-1 for o in b[xl]]

tds = ds.with_transform(transformi)

In [10]:
tds_train, tds_val = tds.get('train'), tds.get('test')
tds_train, tds_val

(Dataset({
     features: ['image', 'label'],
     num_rows: 60000
 }),
 Dataset({
     features: ['image', 'label'],
     num_rows: 10000
 }))

In [12]:
tds_train[xl][0].shape, tds_train[xl][0][:, 15:17, :]

(torch.Size([1, 32, 32]),
 tensor([[[-1.00, -1.00, -1.00, -1.00, -1.00, -1.00, -1.00, -1.00, -1.00, -1.00, -1.00, -0.97, -1.00, -1.00, -0.57,  0.85,  0.79,  0.80,
            0.79,  0.88,  0.82,  0.67,  0.71,  0.75,  0.84,  0.70,  0.70,  0.64, -0.28, -1.00, -1.00, -1.00],
          [-1.00, -1.00, -1.00, -1.00, -0.99, -0.97, -0.95, -0.95, -0.98, -1.00, -1.00, -1.00, -1.00, -1.00,  0.86,  0.77,  0.70,  0.75,
            0.74,  0.72,  0.74,  0.73,  0.69,  0.75,  0.80,  0.69,  0.71,  1.00, -0.40, -1.00, -1.00, -1.00]]]))

In [13]:
dls = DataLoaders.from_dd(tds, bs, num_workers=fc.defaults.cpus)

In [ ]:
# method 1
batch = next(iter(dls.train))
[o.shape for o in batch]

[torch.Size([512, 1, 32, 32]), torch.Size([512])]

In [40]:
# method 2
for batch in dls.train:
    x, y = batch
    print(f'x.shape: {x.shape}, y.shape: {y.shape}')
    break

x.shape: torch.Size([512, 1, 32, 32]), y.shape: torch.Size([512])
